# PAYBACK Lightweight Assistant — Performance Report
Load test against the live Cloud Run deployment `https://payback-assistant-506488371374.europe-west3.run.app` (1 vCPU / 512 MiB per instance, scale-to-zero, max 3 instances, concurrency 80).

In [1]:
import json, statistics, time, uuid, requests
from concurrent.futures import ThreadPoolExecutor
BASE = 'https://payback-assistant-506488371374.europe-west3.run.app'
QUERIES = ['günstige Windeln', 'Suche Shampoo bei dm', 'pasta tomaten parmesan',
           'Schokolade und Chips', 'Mineralwasser 6er']  # rules path = steady-state hot path
def run_load(n_requests, concurrency):
    latencies, errors = [], 0
    session = requests.Session()
    def one(i):
        nonlocal errors
        t0 = time.perf_counter()
        try:
            r = session.post(f'{BASE}/assist', json={'query': QUERIES[i % len(QUERIES)]}, timeout=30)
            r.raise_for_status()
            latencies.append((time.perf_counter() - t0) * 1000)
        except Exception:
            errors += 1
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        list(pool.map(one, range(n_requests)))
    wall = time.perf_counter() - t0
    latencies.sort()
    p = lambda q: latencies[min(len(latencies) - 1, int(q * len(latencies)))]
    return {'requests': n_requests, 'concurrency': concurrency, 'errors': errors,
            'wall_s': round(wall, 2), 'rps': round(len(latencies) / wall, 1),
            'p50_ms': round(p(.5), 1), 'p95_ms': round(p(.95), 1), 'p99_ms': round(p(.99), 1)}
requests.get(f'{BASE}/health', timeout=30).json()  # warm the instance

{'status': 'ok',
 'version': '1.0.0',
 'products': 97,
 'llm': 'gemini-2.5-flash-lite'}

## Scaling proof: throughput grows with concurrency, tail latency stays bounded

In [2]:
results = [run_load(150, c) for c in (5, 20, 40)]
for r in results:
    print(r)

{'requests': 150, 'concurrency': 5, 'errors': 0, 'wall_s': 4.89, 'rps': 30.7, 'p50_ms': 141.9, 'p95_ms': 159.3, 'p99_ms': 731.8}
{'requests': 150, 'concurrency': 20, 'errors': 0, 'wall_s': 2.53, 'rps': 59.3, 'p50_ms': 139.9, 'p95_ms': 1555.4, 'p99_ms': 1569.4}
{'requests': 150, 'concurrency': 40, 'errors': 0, 'wall_s': 3.13, 'rps': 47.9, 'p50_ms': 146.9, 'p95_ms': 2828.0, 'p99_ms': 2848.7}


## LLM model tiers: server-side latency per Gemini model
Same paraphrase queries, `llm_mode: always`, measured via the response's own `latency_ms` (server-side handler time incl. the Vertex AI call — network round-trip to the client excluded).

**LLM used** counts responses whose `engine` shows the model actually answered; the remainder hit the service's 6 s LLM timeout (plus one retry) and fell back to the deterministic rules path — the ~12 s latencies of slow tiers are the two exhausted timeouts, not usable inference. `always` means *always attempt* the LLM; the timeout guard means the API still answers when a tier can't keep up.

In [3]:
MODELS = ['gemini-2.5-flash-lite', 'gemini-2.5-flash', 'gemini-3.1-flash-lite', 'gemini-3.5-flash']
PARAPHRASES = ['etwas gegen wunden Po beim Kleinkind', 'pancakes for the kids',
               'was Schönes zum Naschen', 'stuff for taco night', 'spiegelei fürs frühstück',
               'something for a rainy sunday afternoon']
print(f"{'model':26} {'p50 ms':>8} {'p95 ms':>8} {'mean ms':>8} {'LLM used':>9}  verdict")
nonce = uuid.uuid4().hex[:6]  # cache-buster: benchmark must measure inference, not the response cache
for m in MODELS:
    lats, used = [], 0
    for q in PARAPHRASES:
        r = requests.post(f'{BASE}/assist', json={'query': f'{q} {nonce}', 'llm_mode': 'always',
                          'model': m, 'user_id': f'tier-{m}'}, timeout=60).json()
        lats.append(r['latency_ms'])
        used += m in r['engine']  # engine names the model only when it answered
    lats.sort()
    n = len(PARAPHRASES)
    verdict = 'ok' if used == n else (f'{n - used} timeout fallback(s) to rules')
    print(f'{m:26} {lats[len(lats)//2]:8.0f} {lats[-1]:8.0f} {statistics.mean(lats):8.0f} '
          f'{used}/{n:>3}  {verdict}')

model                        p50 ms   p95 ms  mean ms  LLM used  verdict


gemini-2.5-flash-lite           372      435      381 6/  6  ok


gemini-2.5-flash                708      898      684 6/  6  ok


gemini-3.1-flash-lite          1203     1266     1168 6/  6  ok


gemini-3.5-flash               4552     5040     4219 0/  6  6 timeout fallback(s) to rules


### Response cache: repeated queries are free
A small TTL cache keyed by (model, normalized query, **context**) — same query with the same context skips inference entirely. The demo uses two fresh users (identical empty context): the cache is deliberately context-aware, so the *same* user's repeat after their profile changed is a different key — a personalization-correctness property, not a cache miss bug. One cookie session pins both calls to one instance.

In [4]:
s = requests.Session()  # cookie session -> Cloud Run affinity pins both calls to one instance
q = f'ideen für einen filmabend {uuid.uuid4().hex[:6]}'
r1 = s.post(f'{BASE}/assist', json={'query': q,
                   'llm_mode': 'always', 'user_id': f'cache-a-{uuid.uuid4().hex[:6]}'}, timeout=60).json()
r2 = s.post(f'{BASE}/assist', json={'query': q,
                   'llm_mode': 'always', 'user_id': f'cache-b-{uuid.uuid4().hex[:6]}'}, timeout=60).json()
print(f"first call : {r1['latency_ms']:.0f} ms  ({r1['engine']})")
print(f"repeat call: {r2['latency_ms']:.0f} ms  (served from cache)")

first call : 391 ms  (rules+gemini-2.5-flash-lite@vertex-ai)
repeat call: 0 ms  (served from cache)


## Cost per 1000 requests (measured, Cloud Run request-based billing)
vCPU $0.000024/s + memory $0.0000025/GiB·s billed only while serving, plus $0.40 per million requests. LLM-escalated requests add Gemini 2.5 Flash-Lite on Vertex AI (~$0.10/M input + $0.40/M output tokens; ~700 in / 60 out per call).

In [5]:
best = max(results, key=lambda r: r['rps'])
per_req_s = 1 / best['rps']
infra = (0.000024 + 0.0000025 * 0.5) * per_req_s + 0.40 / 1_000_000
llm_call = 700 / 1e6 * 0.10 + 60 / 1e6 * 0.40   # per escalated call
for esc in (0.0, 0.10, 0.25):
    total = (infra + esc * llm_call) * 1000
    print(f'cost per 1000 requests at {esc:.0%} LLM escalation: ${total:.4f}')

cost per 1000 requests at 0% LLM escalation: $0.0008
cost per 1000 requests at 10% LLM escalation: $0.0102
cost per 1000 requests at 25% LLM escalation: $0.0243


## How prompt/inference time was optimized
**Architecture level** — inference is removed from the hot path rather than optimized: deterministic rules + concept-grouped BM25 answer every known-vocabulary query in ~1–2 ms; the LLM runs only on unknown terms (or `llm_mode: always`), and recurring misses can be promoted into the synonym lexicon, shrinking the escalation rate over time.

**Inference level** — measured on the live service, each lever verified by before/after benchmarks:
1. **`thinkingBudget: 0`** for the Gemini 2.5 family: these models spend hidden reasoning tokens by default, useless for classification. Effect: gemini-2.5-flash p50 **4 287 → 773 ms (−82%)** and its timeout fallbacks disappeared (2/6 → 0/6).
2. **Compact wire schema + `maxOutputTokens: 96`**: single-letter JSON keys (`{"i","l","p","t","c"}`) roughly halve decode tokens — decode time dominates generation latency. Combined with a ~60% smaller prompt (terse rules + few-shot examples, context profile as one line): flash-lite p50 **683 → 401 ms**.
3. **Regional Vertex endpoint** (`europe-west4`, adjacent to the Cloud Run region) for the 2.5 family: ~250 ms RTT saved vs the global endpoint (3.x models are global-only).
4. **TTL response cache** on (model, query, context): repeated queries cost 0 ms.
5. **Hard 6 s timeout + one retry, falling back to rules**: the API never blocks on a slow model — gemini-3.5-flash still cannot meet the budget (its floor is ~3 s+ and `maxOutputTokens` truncates its mandatory thinking), so it fails fast to rules; the tier table above documents this honestly.

`responseMimeType: application/json` + temperature 0 eliminate parse retries; the index is built once at startup from a catalog baked into the image; the service is stateless (Cloud Run, 3 × concurrency 80, session affinity for context locality).